# JSON-to-Payer-PDF Filling Workflow

This notebook demonstrates the product workflow for filling non-editable payer PDF forms from a stable database JSON payload.

The core idea is intentionally simple:

```text
PDF template + DB schema/value JSON + reviewed mapping artifact -> flattened filled PDF
```

This means the final production fill does **not** depend on editable AcroForms, manual typing, or GPT calls at runtime. The form is filled by drawing formatted values directly onto the original PDF in validated coordinates.

## Who this notebook is for

- Engineering teams evaluating the implementation and data model.
- Product/business teams reviewing the operational workflow.
- Implementation teams onboarding a new payer template.
- QA teams checking why a field was filled, skipped, or flagged for review.

## Operating principles

- Prefer blank over wrong: unsafe fields are skipped.
- GPT may help create or maintain mappings, but GPT never invents fill coordinates.
- Every production draw operation requires `review == "ok"`.
- Values are formatted deterministically before drawing.
- Fill rectangles are sanitized so values do not overwrite labels, helper text, or instructions.
- The output is a flattened PDF that preserves the original page structure.

## Notebook flow

Run the notebook top-to-bottom for a first demonstration. After that, individual sections can be rerun independently when creating a mapping, reviewing diagnostics, or filling a new JSON payload against an existing reviewed mapping.

## 1. Environment And Dependencies

This cell defines all imports, constants, data models, and basic JSON/path helpers used by the workflow.

### What is defined here

- Standard Python imports for JSON, hashing, paths, temporary files, dates, and HTTP requests.
- PDF dependencies:
  - `pypdf` for reading, writing, merging, and preserving PDFs.
  - `reportlab` for drawing flattened text and checkbox marks.
  - Optional `fitz`/PyMuPDF support when available for richer text extraction.
- Constants controlling date parsing and layout safety.
- Data models:
  - `TextRun`: a positioned piece of PDF text.
  - `RectCandidate`: a possible form field or checkbox candidate.
- Utility helpers:
  - JSON load/write.
  - `slug` for stable field IDs.
  - OpenAI environment loading and JSON response handling.
  - JSON schema path extraction.
  - Safe JSON path resolution.

### Why this matters

The notebook is self-contained for onboarding. Your team can inspect the actual implementation directly in the notebook instead of treating the product as a black box. Pydantic is used at the product boundary so malformed mapping artifacts fail early with readable validation errors.

### Expected output

This cell normally produces no output. If it fails, install or activate the Python environment that includes `pypdf` and `reportlab`.

In [ ]:
"""
Template-driven JSON-to-PDF filling engine.

This module creates reviewable template mapping artifacts and fills payer PDF
forms deterministically from the stable database JSON schema. It intentionally
does not depend on converting forms to AcroForms.
"""

from __future__ import annotations

import argparse
import base64
import hashlib
import json
import os
import re
import tempfile
from datetime import datetime
from pathlib import Path
from textwrap import wrap
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator
from pypdf import PdfReader, PdfWriter
from reportlab.lib.colors import black
from reportlab.pdfgen import canvas
from reportlab.pdfbase.pdfmetrics import stringWidth


try:
    import fitz  # type: ignore
except Exception:  # pragma: no cover - optional production dependency
    fitz = None


DATE_INPUT_FORMATS = ["%Y%m%d", "%m/%d/%Y", "%Y-%m-%d", "%m-%d-%Y"]
LABEL_VALUE_GAP = 18
FIELD_RIGHT_PADDING = 4
MIN_FILL_WIDTH = 24


def validate_rect_value(value: Any) -> list[float]:
    if not isinstance(value, list) or len(value) != 4:
        raise ValueError("rect must be a list of four numbers")
    try:
        rect = [float(item) for item in value]
    except (TypeError, ValueError) as exc:
        raise ValueError("rect values must be numeric") from exc
    if rect[2] <= rect[0] or rect[3] <= rect[1]:
        raise ValueError("rect must have positive width and height")
    return rect


class PdfFillModel(BaseModel):
    model_config = ConfigDict(extra="allow")


class TextRun(PdfFillModel):
    text: str
    page: int
    x: float
    y: float
    size: float

    @property
    def x2(self) -> float:
        return self.x + stringWidth(self.text, "Helvetica", self.size)


class RectCandidate(PdfFillModel):
    page: int
    source: str
    label: str
    labelRect: list[float] | None
    fillRect: list[float]
    sectionPath: list[str]
    nearbyText: str
    confidence: float
    kind: str = "text"

    @field_validator("labelRect")
    @classmethod
    def validate_optional_rect(cls, value: Any) -> list[float] | None:
        return None if value is None else validate_rect_value(value)

    @field_validator("fillRect")
    @classmethod
    def validate_fill_rect(cls, value: Any) -> list[float]:
        return validate_rect_value(value)


class CheckboxOption(PdfFillModel):
    value: str = "true"
    label: str = ""
    boxRect: list[float]
    page: int | None = None

    @field_validator("boxRect")
    @classmethod
    def validate_box_rect(cls, value: Any) -> list[float]:
        return validate_rect_value(value)


class MappingField(PdfFillModel):
    fieldId: str
    type: str = "text"
    jsonPath: str | None = None
    jsonPathSecondary: str | None = None
    page: int
    sectionPath: list[str] = Field(default_factory=list)
    label: str = ""
    labelRect: list[float] | None = None
    fillRect: list[float]
    fontSize: float = 9
    overflow: str = "shrink_then_clip"
    confidence: float | None = None
    nearbyText: str = ""
    review: str = "needs_review"
    valueTransform: str | None = None
    format: str | None = None
    mode: str | None = None
    separators: str | None = None
    currencySymbol: bool = False

    @field_validator("labelRect")
    @classmethod
    def validate_label_rect(cls, value: Any) -> list[float] | None:
        return None if value is None else validate_rect_value(value)

    @field_validator("fillRect")
    @classmethod
    def validate_field_rect(cls, value: Any) -> list[float]:
        return validate_rect_value(value)


class CheckboxGroup(PdfFillModel):
    fieldId: str
    type: str = "checkbox_group"
    jsonPath: str | None = None
    page: int | None = None
    sectionPath: list[str] = Field(default_factory=list)
    label: str = ""
    options: list[CheckboxOption] = Field(default_factory=list)
    confidence: float | None = None
    review: str = "needs_json_path"
    multiSelect: bool = False
    matchMode: str = "contains_normalized"


class TemplateMapping(PdfFillModel):
    templateId: str
    templateVersion: str = "v1"
    templateFingerprint: dict[str, Any] = Field(default_factory=dict)
    pageCount: int
    pageSizes: list[list[float]] = Field(default_factory=list)
    fields: list[MappingField] = Field(default_factory=list)
    checkboxGroups: list[CheckboxGroup] = Field(default_factory=list)
    dateFields: list[MappingField] = Field(default_factory=list)
    repeaters: list[dict[str, Any]] = Field(default_factory=list)
    createdBy: str = "pdf_fill_engine.create_mapping"
    reviewStatus: str = "draft_needs_review"
    schemaPaths: list[str] = Field(default_factory=list)
    candidateCount: int = 0
    mappingMode: str | None = None


class FilledFieldDiagnostic(PdfFillModel):
    fieldId: str | None = None
    type: str
    jsonPath: str | None = None
    renderedValuePreview: str
    renderedRect: list[float] | None = None
    truncated: bool | None = None

    @field_validator("renderedRect")
    @classmethod
    def validate_rendered_rect(cls, value: Any) -> list[float] | None:
        return None if value is None else validate_rect_value(value)


class SkippedFieldDiagnostic(PdfFillModel):
    fieldId: str
    reason: str
    jsonPath: str | None = None


class FillDiagnostics(PdfFillModel):
    filledFields: list[FilledFieldDiagnostic] = Field(default_factory=list)
    skippedFields: list[SkippedFieldDiagnostic] = Field(default_factory=list)
    skippedReasons: dict[str, int] = Field(default_factory=dict)
    warnings: list[str] = Field(default_factory=list)


class FillResult(PdfFillModel):
    output: str
    mappingDrift: bool
    fields: int
    dateFields: int
    checkboxGroups: int
    diagnostics: FillDiagnostics


class MappingValidationResult(PdfFillModel):
    errors: list[str] = Field(default_factory=list)
    warnings: list[str] = Field(default_factory=list)


def to_jsonable(payload: Any) -> Any:
    if isinstance(payload, BaseModel):
        return payload.model_dump(mode="json", exclude_none=False)
    if isinstance(payload, list):
        return [to_jsonable(item) for item in payload]
    if isinstance(payload, dict):
        return {key: to_jsonable(value) for key, value in payload.items()}
    return payload


def normalize_mapping_payload(mapping: Any) -> dict[str, Any]:
    return TemplateMapping.model_validate(mapping).model_dump(mode="json", exclude_none=False)


def pydantic_error_messages(exc: ValidationError) -> list[str]:
    messages = []
    for error in exc.errors():
        location = ".".join(str(part) for part in error.get("loc", []))
        prefix = f"{location}: " if location else ""
        messages.append(prefix + str(error.get("msg", "validation error")))
    return messages


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_jsonable(payload), indent=2, ensure_ascii=False), encoding="utf-8")


def slug(text: str, fallback: str = "field") -> str:
    value = re.sub(r"[^a-zA-Z0-9]+", "_", text.lower()).strip("_")
    value = re.sub(r"_+", "_", value)
    return value[:80] or fallback


def load_env_value(name: str, env_path: Path | None = None) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    for path in [p for p in [env_path, Path("work/.env"), Path(".env")] if p]:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw = line.split("=", 1)
            if key.strip() == name:
                value = raw.strip().strip('"').strip("'")
                if value:
                    return value
    raise RuntimeError(f"{name} was not found in the environment or local .env.")


def openai_json_response(
    pdf_path: Path,
    prompt: str,
    schema: dict[str, Any],
    model: str,
    env_path: Path | None = None,
) -> dict[str, Any]:
    api_key = load_env_value("OPENAI_API_KEY", env_path)
    file_data = "data:application/pdf;base64," + base64.b64encode(pdf_path.read_bytes()).decode("ascii")
    body = {
        "model": model,
        "input": [
            {
                "role": "user",
                "content": [
                    {"type": "input_file", "filename": pdf_path.name, "file_data": file_data},
                    {"type": "input_text", "text": prompt},
                ],
            }
        ],
        "text": {
            "format": {
                "type": "json_schema",
                "name": "pdf_template_mapping",
                "schema": schema,
                "strict": True,
            }
        },
    }
    request = Request(
        "https://api.openai.com/v1/responses",
        data=json.dumps(body).encode("utf-8"),
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urlopen(request, timeout=180) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI API request failed with HTTP {exc.code}: {detail}") from exc
    except URLError as exc:
        raise RuntimeError(f"OpenAI API request failed: {exc.reason}") from exc

    chunks: list[str] = []

    def walk(value: Any) -> None:
        if isinstance(value, dict):
            if value.get("type") in {"output_text", "text"} and isinstance(value.get("text"), str):
                chunks.append(value["text"])
            for child in value.values():
                walk(child)
        elif isinstance(value, list):
            for child in value:
                walk(child)

    if isinstance(payload.get("output_text"), str):
        text = payload["output_text"]
    else:
        walk(payload.get("output", []))
        text = "\n".join(chunks).strip()
    return json.loads(text)


def extract_json_paths(data: Any, prefix: str = "") -> list[str]:
    paths: list[str] = []
    if isinstance(data, dict):
        for key, value in data.items():
            child = f"{prefix}.{key}" if prefix else key
            paths.extend(extract_json_paths(value, child))
    elif isinstance(data, list):
        child = f"{prefix}[]"
        if data:
            paths.extend(extract_json_paths(data[0], child))
        else:
            paths.append(child)
    else:
        paths.append(prefix)
    return paths


def is_nonempty_json_path(path: Any) -> bool:
    return isinstance(path, str) and bool(path.strip())


def resolve_json_path(data: Any, path: str | None) -> Any:
    if not is_nonempty_json_path(path):
        return None
    current = data
    for part in str(path).strip().split("."):
        if not part:
            return None
        if part.endswith("[]"):
            key = part[:-2]
            if not key:
                return None
            current = current.get(key) if isinstance(current, dict) else None
            if not isinstance(current, list):
                return None
        elif isinstance(current, dict):
            if part not in current:
                return None
            current = current.get(part)
        else:
            return None
        if current is None:
            return None
    return current


## 2. Inputs

Configure the PDF form and DB JSON payload here.

### Required inputs

- `PDF_PATH`: the payer form template to process.
- `DATA_JSON_PATH`: the DB schema/value JSON payload. The schema is expected to remain stable under `result`.

### Generated outputs

- `MAPPING_PATH`: draft or reviewed template mapping artifact.
- `FILLED_PDF_PATH`: flattened filled PDF output.

### Mapping mode

- `USE_LLM_MAPPING = False`: deterministic heuristic mapping only. This is best for local demos and offline runs.
- `USE_LLM_MAPPING = True`: GPT can select candidate fields and schema paths. Coordinates still come only from PDF geometry. GPT does not create or modify rectangles.

### Credential safety

If GPT mode is enabled, the notebook reads `OPENAI_API_KEY` from environment variables or a local `.env` file. Do not paste API keys into notebook cells.

### How to use this cell

Change only the path variables for a new form. Keep the output naming convention while onboarding so artifacts are easy to compare.

## 1A. Forward Safety Helpers For Geometry

This small helper cell exists because layout extraction uses rectangle validation before the full runtime section is introduced later in the notebook.

### What is defined here

- `preview_value`: creates short diagnostic previews without dumping large values.
- `valid_rect`: confirms a rectangle has four numeric coordinates and positive width/height.

### Why this is separate

The notebook is organized as a step-by-step teaching flow. Candidate extraction happens before the full fill runtime is introduced, so these small helpers are available early.

### Expected output

No output. These functions are used by later layout cells.

In [ ]:
def preview_value(value: Any, limit: int = 120) -> str:
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if len(text) <= limit else text[: limit - 3] + "..."


def valid_rect(rect: Any) -> bool:
    if not isinstance(rect, list) or len(rect) != 4:
        return False
    try:
        x1, y1, x2, y2 = [float(v) for v in rect]
    except (TypeError, ValueError):
        return False
    return x2 > x1 and y2 > y1


In [ ]:
# Choose the form and DB JSON payload to run through the notebook.
PDF_PATH = Path("ANTHEM_CLAIMS_NV_00001.pdf")
DATA_JSON_PATH = Path("database_schema.json")

# Mapping and output artifacts produced by this notebook.
TEMPLATE_ID = slug(PDF_PATH.stem, "template") + "_notebook_demo"
MAPPING_PATH = Path("outputs") / f"{PDF_PATH.stem}_notebook_mapping.json"
FILLED_PDF_PATH = Path("outputs") / f"{PDF_PATH.stem}_notebook_filled.pdf"

# Set True only when you want GPT to help map candidates to schema paths.
# Coordinates still come from PDF geometry; GPT never invents fill positions.
USE_LLM_MAPPING = False
OPENAI_MODEL = "gpt-4.1-mini"
ENV_FILE = Path("work/.env")

print("PDF:", PDF_PATH.resolve())
print("Data JSON:", DATA_JSON_PATH.resolve())
print("Mapping output:", MAPPING_PATH.resolve())
print("Filled PDF output:", FILLED_PDF_PATH.resolve())

## 3. Load Data And Inspect Schema Paths

This stage loads the DB JSON payload and extracts all allowed scalar/list paths.

### Why schema paths matter

The mapper is constrained to known paths such as:

- `result.patient.firstName`
- `result.subscriber.memberID`
- `result.provider.facilityName`
- `result.claim.claimNumber`
- `result.claim.serviceLines[]...`

This prevents a heuristic or GPT-assisted mapping step from inventing fields that do not exist in the source data.

### What to inspect

- Confirm that expected sections exist: `patient`, `subscriber`, `provider`, `payer`, `claim`.
- Confirm that important claim paths exist, especially dates, amounts, claim number, and service lines.
- If a field cannot be mapped later, first check whether the source JSON actually contains the needed value.

### Expected output

A count of extracted paths and the first several schema paths.

In [ ]:
data = load_json(DATA_JSON_PATH)
schema_paths = extract_json_paths(data)

print(f"Loaded {len(schema_paths)} scalar/list schema paths")
for path in schema_paths[:30]:
    print("-", path)

## 4. PDF Fingerprint And Layout Extraction

This stage reads the PDF template and extracts the layout signals needed for mapping.

### What this stage extracts

- Page count and page sizes.
- Normalized text fingerprint for drift detection.
- Text runs with page/x/y/font-size information.
- Vector rectangles/lines where available.
- Section-aware field candidates.

### Candidate detection strategy

The engine identifies candidates from labels and geometry:

- Text labels ending in `:` are treated as possible form fields.
- Nearby section headers are attached as `sectionPath`.
- Checkbox-sized rectangles are detected separately.
- Larger response boxes are detected for narrative fields such as `Explain`, `Reason`, or `Comments`.
- Fill rectangles are sanitized so they start after labels and avoid existing printed helper text.

### What to inspect

Look at the printed candidate list. For a good mapping, true business fields should appear with sensible labels, pages, and rectangles. Instruction text may appear as candidates but should later be marked `review != "ok"`.

### Expected output

A layout summary and sample candidates.

In [ ]:
def pdf_text(reader: PdfReader) -> str:
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def compute_fingerprint(pdf_path: Path) -> dict[str, Any]:
    reader = PdfReader(str(pdf_path))
    page_sizes = [
        [float(page.mediabox.width), float(page.mediabox.height)]
        for page in reader.pages
    ]
    text = pdf_text(reader)
    digest = hashlib.sha256()
    digest.update(str(page_sizes).encode("utf-8"))
    digest.update(re.sub(r"\s+", " ", text).encode("utf-8"))
    return {
        "algorithm": "sha256(page_sizes+normalized_text)",
        "value": digest.hexdigest(),
        "pageCount": len(reader.pages),
        "pageSizes": page_sizes,
        "textLength": len(text),
    }


def extract_text_runs_pypdf(reader: PdfReader) -> list[TextRun]:
    runs: list[TextRun] = []

    for page_index, page in enumerate(reader.pages):
        def visitor(text, cm, tm, font, size):
            clean = re.sub(r"\s+", " ", text or "").strip()
            if clean:
                runs.append(
                    TextRun(
                        text=clean,
                        page=page_index + 1,
                        x=float(tm[4]),
                        y=float(tm[5]),
                        size=float(size),
                    )
                )

        page.extract_text(visitor_text=visitor)
    return runs


def extract_text_runs_pymupdf(pdf_path: Path) -> list[TextRun]:
    if fitz is None:
        return []
    runs: list[TextRun] = []
    doc = fitz.open(str(pdf_path))
    for page_index, page in enumerate(doc):
        height = float(page.rect.height)
        for block in page.get_text("dict").get("blocks", []):
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    text = re.sub(r"\s+", " ", span.get("text", "")).strip()
                    if not text:
                        continue
                    x0, y0, _x1, y1 = span["bbox"]
                    runs.append(
                        TextRun(
                            text=text,
                            page=page_index + 1,
                            x=float(x0),
                            y=round(height - float(y1), 2),
                            size=float(span.get("size", 9)),
                        )
                    )
    return runs


def extract_rectangles(reader: PdfReader) -> list[dict[str, Any]]:
    rects: list[dict[str, Any]] = []
    for page_index, page in enumerate(reader.pages):
        def before(op, args, cm, tm):
            op = op.decode() if isinstance(op, bytes) else op
            if op == "re" and len(args) >= 4:
                x, y, w, h = [round(float(v), 2) for v in args[:4]]
                if w > 0 and h > 0:
                    rects.append({"page": page_index + 1, "rect": [x, y, x + w, y + h]})

        page.extract_text(visitor_operand_before=before)
    return rects


def group_lines(runs: list[TextRun], tolerance: float = 3.0) -> list[list[TextRun]]:
    groups: list[list[TextRun]] = []
    for run in sorted(runs, key=lambda r: (r.page, -r.y, r.x)):
        for group in groups:
            if group[0].page == run.page and abs(group[0].y - run.y) <= tolerance:
                group.append(run)
                break
        else:
            groups.append([run])
    return [sorted(group, key=lambda r: r.x) for group in groups]


def line_text(line: list[TextRun]) -> str:
    return re.sub(r"\s+", " ", " ".join(run.text for run in line)).strip()


def infer_section_path(line: list[TextRun], current: list[str]) -> list[str]:
    text = line_text(line)
    lower = text.lower()
    if not text:
        return current
    section_terms = [
        "information",
        "claim",
        "provider",
        "subscriber",
        "patient",
        "requestor",
        "reason",
        "supporting documentation",
        "section",
    ]
    is_section = (
        any(term in lower for term in section_terms)
        and len(text) <= 90
        and ":" not in text
        and not re.match(r"^\d+\.", text)
    )
    if is_section:
        return [text]
    return current


def nearby_text(runs: list[TextRun], page: int, rect: list[float], radius: float = 35) -> str:
    x1, y1, x2, y2 = rect
    selected = [
        r
        for r in runs
        if r.page == page
        and x1 - radius <= r.x <= x2 + radius
        and y1 - radius <= r.y <= y2 + radius
    ]
    return re.sub(r"\s+", " ", " ".join(r.text for r in sorted(selected, key=lambda r: (-r.y, r.x))))[:500]


def run_rect(run: TextRun) -> list[float]:
    return [run.x, run.y - 2, run.x2, run.y + run.size + 2]


def vertical_overlap(a: list[float], b: list[float]) -> float:
    return max(0, min(a[3], b[3]) - max(a[1], b[1]))


def rect_intersects(a: list[float], b: list[float]) -> bool:
    return max(a[0], b[0]) < min(a[2], b[2]) and max(a[1], b[1]) < min(a[3], b[3])


def protected_text_runs(runs: list[TextRun], page: int, rect: list[float], label_rect: list[float] | None = None) -> list[TextRun]:
    protected: list[TextRun] = []
    rect_height = max(1, rect[3] - rect[1])
    for run in runs:
        if run.page != page:
            continue
        candidate = run_rect(run)
        if label_rect and rect_intersects(candidate, label_rect):
            continue
        overlap = vertical_overlap(candidate, rect)
        if overlap <= 0 or overlap / min(rect_height, max(1, candidate[3] - candidate[1])) < 0.35:
            continue
        if candidate[2] <= rect[0] + 1 or candidate[0] >= rect[2] - 1:
            continue
        protected.append(run)
    return sorted(protected, key=lambda item: item.x)


def sanitize_fill_rect(field: dict[str, Any], page_width: float, runs: list[TextRun] | None = None) -> tuple[list[float] | None, list[str]]:
    warnings: list[str] = []
    rect = field.get("fillRect")
    if not valid_rect(rect):
        return None, ["invalid_fill_rect"]
    x1, y1, x2, y2 = [float(v) for v in rect]
    label_rect = field.get("labelRect")
    if valid_rect(label_rect):
        label_x1, label_y1, label_x2, label_y2 = [float(v) for v in label_rect]
        min_x = min(page_width - MIN_FILL_WIDTH, label_x2 + LABEL_VALUE_GAP)
        if x1 < min_x:
            x1 = min_x
            warnings.append("fillRect_shifted_after_label")
        y1 = max(y1, label_y1 - 1)
        y2 = min(max(y2, y1 + 10), label_y2 + 3)
    x1 = max(0, x1)
    x2 = min(page_width - FIELD_RIGHT_PADDING, x2)
    if runs:
        for run in protected_text_runs(runs, int(field.get("page") or 0), [x1, y1, x2, y2], label_rect):
            blocker = run_rect(run)
            if blocker[0] > x1 + 1:
                new_x2 = min(x2, blocker[0] - FIELD_RIGHT_PADDING)
                if new_x2 < x2:
                    x2 = new_x2
                    warnings.append(f"fillRect_trimmed_before_pdf_text:{preview_value(run.text, 40)}")
                break
            if blocker[2] >= x1:
                x1 = blocker[2] + FIELD_RIGHT_PADDING
                warnings.append(f"fillRect_shifted_after_pdf_text:{preview_value(run.text, 40)}")
    if x2 - x1 < MIN_FILL_WIDTH:
        warnings.append("fillRect_too_narrow_after_sanitization")
        return None, warnings
    return [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)], warnings


def build_candidates(pdf_path: Path) -> list[RectCandidate]:
    reader = PdfReader(str(pdf_path))
    runs = extract_text_runs_pymupdf(pdf_path) or extract_text_runs_pypdf(reader)
    candidates: list[RectCandidate] = []
    current_section_by_page: dict[int, list[str]] = {}

    for line in group_lines(runs):
        page = line[0].page
        current = current_section_by_page.get(page, [])
        current = infer_section_path(line, current)
        current_section_by_page[page] = current
        text = line_text(line)
        if ":" not in text:
            continue
        colon_runs = [i for i, r in enumerate(line) if ":" in r.text]
        segment_start = 0
        for pos, colon_index in enumerate(colon_runs):
            label_runs = line[segment_start : colon_index + 1]
            if not label_runs:
                continue
            label = line_text(label_runs)
            label_x1 = min(r.x for r in label_runs)
            label_y1 = min(r.y for r in label_runs) - 2
            label_x2 = max(r.x2 for r in label_runs)
            label_y2 = max(r.y + r.size for r in label_runs)
            next_x = float(reader.pages[page - 1].mediabox.width) - 36
            if pos + 1 < len(colon_runs):
                next_colon = colon_runs[pos + 1]
                next_x = line[max(colon_index + 1, next_colon - 2)].x - 8
            fill_x1 = min(label_x2 + LABEL_VALUE_GAP, next_x - 35)
            fill_x2 = next_x
            if fill_x2 - fill_x1 >= 28:
                raw_fill = [round(fill_x1, 2), round(label_y1, 2), round(fill_x2, 2), round(label_y1 + 16, 2)]
                label_rect = [round(label_x1, 2), round(label_y1, 2), round(label_x2, 2), round(label_y2, 2)]
                field_stub = {"page": page, "fillRect": raw_fill, "labelRect": label_rect}
                sanitized, _warnings = sanitize_fill_rect(field_stub, float(reader.pages[page - 1].mediabox.width), runs)
                if not sanitized:
                    segment_start = colon_index + 1
                    continue
                fill = sanitized
                candidates.append(
                    RectCandidate(
                        page=page,
                        source="label",
                        label=label,
                        labelRect=label_rect,
                        fillRect=fill,
                        sectionPath=current,
                        nearbyText=nearby_text(runs, page, fill),
                        confidence=0.82,
                    )
                )
            segment_start = colon_index + 1

    for item in extract_rectangles(reader):
        page = item["page"]
        x1, y1, x2, y2 = item["rect"]
        w = x2 - x1
        h = y2 - y1
        if 5 <= w <= 18 and 5 <= h <= 18 and abs(w - h) <= 4:
            fill = [x1, y1, x2, y2]
            candidates.append(
                RectCandidate(
                    page=page,
                    source="checkbox",
                    label="checkbox",
                    labelRect=None,
                    fillRect=fill,
                    sectionPath=current_section_by_page.get(page, []),
                    nearbyText=nearby_text(runs, page, fill, radius=60),
                    confidence=0.85,
                    kind="checkbox",
                )
            )
        elif w >= 200 and h >= 60:
            fill = [x1 + 3, y1 + 3, x2 - 3, y2 - 3]
            context = nearby_text(runs, page, fill, radius=70)
            if any(term in context.lower() for term in ["explain", "reason", "comments", "other"]):
                candidates.append(
                    RectCandidate(
                        page=page,
                        source="response_box",
                        label="free text",
                        labelRect=None,
                        fillRect=fill,
                        sectionPath=current_section_by_page.get(page, []),
                        nearbyText=context,
                        confidence=0.7,
                    )
                )

    return dedupe_candidates(candidates)


def rect_overlap(a: list[float], b: list[float]) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x = max(0, min(ax2, bx2) - max(ax1, bx1))
    y = max(0, min(ay2, by2) - max(ay1, by1))
    intersection = x * y
    area = min(max(1, (ax2 - ax1) * (ay2 - ay1)), max(1, (bx2 - bx1) * (by2 - by1)))
    return intersection / area


def dedupe_candidates(candidates: list[RectCandidate]) -> list[RectCandidate]:
    rank = {"label": 0, "checkbox": 0, "response_box": 2, "azure": 3}
    kept: list[RectCandidate] = []
    for cand in sorted(candidates, key=lambda c: (c.page, rank.get(c.source, 9), -c.confidence)):
        if any(cand.kind == other.kind and cand.page == other.page and rect_overlap(cand.fillRect, other.fillRect) > 0.45 for other in kept):
            continue
        kept.append(cand)
    return sorted(kept, key=lambda c: (c.page, -c.fillRect[1], c.fillRect[0]))


In [ ]:
fingerprint = compute_fingerprint(PDF_PATH)
reader = PdfReader(str(PDF_PATH))
text_runs = extract_text_runs_pymupdf(PDF_PATH) or extract_text_runs_pypdf(reader)
rectangles = extract_rectangles(reader)
candidates = build_candidates(PDF_PATH)

print(json.dumps({
    "pageCount": fingerprint["pageCount"],
    "pageSizes": fingerprint["pageSizes"],
    "textLength": fingerprint["textLength"],
    "textRuns": len(text_runs),
    "rectangles": len(rectangles),
    "fieldCandidates": len(candidates),
}, indent=2))

for i, cand in enumerate(candidates[:20], 1):
    print(i, cand.source, cand.kind, cand.page, cand.label, cand.fillRect, cand.sectionPath)

## 5. Mapping Creation

This stage creates a draft `template_mapping.json` artifact for the PDF template.

### Mapping artifact contents

The mapping stores:

- `templateId`, `templateVersion`, and `templateFingerprint`.
- `fields`: normal text fields.
- `dateFields`: date and date-range fields.
- `checkboxGroups`: grouped selection fields.
- `repeaters`: service-line/table metadata when available.
- `reviewStatus`: draft/review state.
- Per-field metadata such as `label`, `sectionPath`, `labelRect`, `fillRect`, `jsonPath`, `valueTransform`, and `review`.

### Heuristic mode

In heuristic mode, field labels and sections are matched against known schema paths. This is conservative and intentionally leaves uncertain fields as `needs_json_path` or another non-ok review state.

### GPT mode

In GPT mode, GPT receives candidate IDs, section context, and allowed JSON paths. GPT returns selected candidate IDs and schema paths. It does not invent coordinates.

### Important product rule

A draft mapping is not automatically trusted. Production fill draws only `review == "ok"` fields.

### Expected output

A summary showing mapping mode, candidate count, and counts by field type.

In [ ]:
def heuristic_json_path(label: str, section: list[str], schema_paths: list[str]) -> str | None:
    text = " ".join(section + [label]).lower()
    candidates = {
        "provider": {
            "name": "result.provider.facilityName",
            "tin": "result.provider.billingTin",
            "tax": "result.provider.billingTaxID",
            "npi": "result.provider.billingNpi",
            "address": "result.provider.facilityAddress",
            "phone": "result.payer.phone",
            "fax": "result.payer.fax",
        },
        "patient": {
            "name": "result.patient.firstName",
            "id": "result.patient.memberID",
            "dob": "result.patient.dateOfBirth",
        },
        "subscriber": {
            "name": "result.subscriber.firstName",
            "id": "result.subscriber.memberID",
            "dob": "result.subscriber.dateOfBirth",
        },
        "claim": {
            "number": "result.claim.claimNumber",
            "authorization": "result.claim.claimAuthorizationNumber",
            "service": "result.claim.claimServiceDateStart",
            "billed": "result.claim.claimSubmittedCharges",
            "disputed": "result.claim.totalDeniedChargedAmount",
            "amount": "result.claim.claimSubmittedCharges",
            "process": "result.claim.dateReceived",
            "received": "result.claim.dateReceived",
            "paid": "result.claim.claimPaidAmount",
            "denial": "result.claim.denialSummary",
            "explain": "result.claim.denialSummary",
            "reason": "result.claim.denialSummary",
        },
    }
    if "subscriber" in text:
        group = candidates["subscriber"]
    elif "patient" in text or "member" in text:
        group = candidates["patient"]
    elif "provider" in text or "tin" in text or "npi" in text:
        group = candidates["provider"]
    else:
        group = candidates["claim"]

    for key, path in group.items():
        if key in text and path in schema_paths:
            return path
    if "office" in text and "name" in text and "result.provider.facilityName" in schema_paths:
        return "result.provider.facilityName"
    if "practice" in text and "name" in text and "result.provider.facilityName" in schema_paths:
        return "result.provider.facilityName"
    if "provider" in text and "name" in text and "result.provider.facilityName" in schema_paths:
        return "result.provider.facilityName"
    if "provider" in text and ("tin" in text or "#" in text) and "result.provider.billingTin" in schema_paths:
        return "result.provider.billingTin"
    if "billed amount" in text and "result.claim.claimSubmittedCharges" in schema_paths:
        return "result.claim.claimSubmittedCharges"
    if "disputed amount" in text and "result.claim.totalDeniedChargedAmount" in schema_paths:
        return "result.claim.totalDeniedChargedAmount"
    if "process date" in text and "result.claim.dateReceived" in schema_paths:
        return "result.claim.dateReceived"
    if "explain" in text and "result.claim.denialSummary" in schema_paths:
        return "result.claim.denialSummary"
    if "claim" in text and "number" in text and "result.claim.claimNumber" in schema_paths:
        return "result.claim.claimNumber"
    if "date" in text and "service" in text and "result.claim.claimServiceDateStart" in schema_paths:
        return "result.claim.claimServiceDateStart"
    if "email" in text and "result.payer.email" in schema_paths:
        return "result.payer.email"
    return None


def mapping_schema() -> dict[str, Any]:
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "fields": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "candidateId": {"type": "integer"},
                        "fieldId": {"type": "string"},
                        "type": {"type": "string", "enum": ["text", "date"]},
                        "jsonPath": {"type": ["string", "null"]},
                        "format": {"type": ["string", "null"]},
                    },
                    "required": ["candidateId", "fieldId", "type", "jsonPath", "format"],
                },
            },
            "checkboxGroups": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "fieldId": {"type": "string"},
                        "jsonPath": {"type": ["string", "null"]},
                        "options": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "properties": {
                                    "candidateId": {"type": "integer"},
                                    "value": {"type": "string"},
                                    "label": {"type": "string"},
                                },
                                "required": ["candidateId", "value", "label"],
                            },
                        },
                    },
                    "required": ["fieldId", "jsonPath", "options"],
                },
            },
            "repeaters": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "repeaterId": {"type": "string"},
                        "jsonPath": {"type": "string"},
                        "page": {"type": "integer"},
                        "startY": {"type": "number"},
                        "rowHeight": {"type": "number"},
                        "maxRows": {"type": "integer"},
                        "columns": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "properties": {
                                    "jsonPath": {"type": "string"},
                                    "name": {"type": "string"},
                                    "rectOffset": {
                                        "type": "array",
                                        "items": {"type": "number"},
                                        "minItems": 4,
                                        "maxItems": 4,
                                    },
                                    "format": {"type": ["string", "null"]},
                                },
                                "required": ["jsonPath", "name", "rectOffset", "format"],
                            },
                        },
                        "overflow": {"type": "string"},
                    },
                    "required": ["repeaterId", "jsonPath", "page", "startY", "rowHeight", "maxRows", "columns", "overflow"],
                },
            },
        },
        "required": ["fields", "checkboxGroups", "repeaters"],
    }


def llm_select_mapping(pdf_path: Path, candidates: list[RectCandidate], schema_paths: list[str], model: str, env_path: Path | None) -> dict[str, Any]:
    candidate_payload = []
    for i, cand in enumerate(candidates, 1):
        item = cand.model_dump(mode="json")
        item["candidateId"] = i
        candidate_payload.append(item)

    allowed_paths = [p for p in schema_paths if p.startswith("result.")]
    prompt = f"""
Create a draft mapping for filling this payer PDF from the fixed database JSON schema.

Rules:
- Select only fields a user would actually fill. Ignore instruction text, page headers, payer addresses, logos, reply/office-use sections, and legal footers.
- Never invent coordinates. Use candidateId only.
- Same label in different sections must map by section context, for example Patient First Name vs Subscriber First Name.
- Use only jsonPath values from the allowed list. Use null if no safe mapping exists.
- Date fields must use type="date" and format="MM/DD/YYYY".
- Group related checkboxes into checkboxGroups. Long checkbox descriptions should remain labels, not fieldIds.
- Do not map provider identifiers to payer identifiers.
- Do not create repeaters unless the PDF has a clear service-line table.

Allowed JSON paths:
{json.dumps(allowed_paths, indent=2)}

Candidates:
{json.dumps(candidate_payload, indent=2)}
""".strip()
    return openai_json_response(pdf_path, prompt, mapping_schema(), model, env_path)


def unique_id(base: str, seen: dict[str, int]) -> str:
    base = slug(base, "field")
    seen[base] = seen.get(base, 0) + 1
    return base if seen[base] == 1 else f"{base}_{seen[base]}"


def normalize_space(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value)).strip()


def infer_value_transform(label: str, field_id: str = "", json_path: str | None = None, kind: str = "text") -> str:
    text = f"{field_id} {label} {json_path or ''}".lower()
    if kind == "date":
        if "service" in text and "date" in text and ("dates" in text or "(s)" in text or "range" in text):
            return "date_range"
        return "date"
    if "date" in text or "dob" in text or "dos" in text:
        if "service" in text and ("dates" in text or "(s)" in text or "range" in text):
            return "date_range"
        return "date"
    if "amount" in text or "charge" in text or "paid" in text or "balance" in text:
        return "currency"
    if "name" in text and json_path:
        if any(prefix in json_path for prefix in ["result.patient.", "result.subscriber.", "result.dependent."]):
            return "name_full"
    return "text"


def add_transform_metadata(field: dict[str, Any], schema_paths: list[str], kind: str | None = None) -> None:
    transform = infer_value_transform(
        str(field.get("label") or ""),
        str(field.get("fieldId") or ""),
        field.get("jsonPath"),
        kind or str(field.get("type") or "text"),
    )
    field.setdefault("valueTransform", transform)
    if transform == "date_range":
        secondary = "result.claim.claimServiceDateEnd"
        if not field.get("jsonPathSecondary") and secondary in schema_paths:
            field["jsonPathSecondary"] = secondary
        field.setdefault("format", "MM/DD/YYYY")
    elif transform == "date":
        field.setdefault("format", "MM/DD/YYYY")


def apply_runtime_backfills(mapping: dict[str, Any], schema_paths: list[str]) -> None:
    allowed = set(schema_paths)
    for collection in ["fields", "dateFields"]:
        for field in mapping.get(collection, []):
            text = f"{field.get('fieldId') or ''} {field.get('label') or ''} {' '.join(field.get('sectionPath') or [])}".lower()
            if "disputed" in text and "amount" in text and "result.claim.totalDeniedChargedAmount" in allowed:
                if field.get("jsonPath") in {None, "", "result.claim.claimSubmittedCharges"}:
                    field["jsonPath"] = "result.claim.totalDeniedChargedAmount"
                    field["review"] = "ok"
                field["valueTransform"] = "currency"
            elif "date" in text and "service" in text and "result.claim.claimServiceDateStart" in allowed:
                field["jsonPath"] = field.get("jsonPath") or "result.claim.claimServiceDateStart"
                if "result.claim.claimServiceDateEnd" in allowed:
                    field["jsonPathSecondary"] = field.get("jsonPathSecondary") or "result.claim.claimServiceDateEnd"
                field["valueTransform"] = "date_range"
                field["format"] = "MM/DD/YYYY"
            elif "process" in text and "date" in text:
                field["valueTransform"] = "date"
                field["format"] = "MM/DD/YYYY"
            else:
                add_transform_metadata(field, schema_paths, "date" if collection == "dateFields" else str(field.get("type") or "text"))


def is_date_like_path(path: Any) -> bool:
    text = str(path or "").lower()
    return any(term in text for term in ["date", "dob", "service"])


def production_skip_reason(field: dict[str, Any], kind: str = "field") -> str | None:
    label = normalize_space(field.get("label") or "").lower()
    field_id = normalize_space(field.get("fieldId") or "").lower()
    combined = f"{field_id} {label}"
    path = str(field.get("jsonPath") or "")
    if kind == "checkbox_group":
        return None
    static_label_terms = [
        "fax to:",
        "mail to:",
        "send to:",
        "for more information",
        "do not submit",
        "you may submit",
        "our determination",
    ]
    if any(term in label for term in static_label_terms):
        return "non_fillable_static_text"
    if label.startswith("-") and label.count("-") >= 2:
        return "option_list_not_text_field"
    if len(label) > 110 and not any(term in label for term in ["explain", "comments:", "remarks:"]):
        return "long_instruction_not_fill_target"
    instruction_terms = ["you must be specific", "handling of the claim", "hand_ling", "billing code", "reason for contesting"]
    if any(term in combined for term in instruction_terms):
        return "long_instruction_not_fill_target"
    if "signature" in label or field_id.endswith("_signature"):
        if "signature" not in path.lower():
            return "unsafe_signature_mapping"
    if "attachment" in label or "attachments" in label or "submitted with your appeal" in label:
        return "non_text_attachment_prompt"
    if "provide more information" in label:
        path_lower = path.lower()
        if not any(term in path_lower for term in ["summary", "description", "reason", "note", "comment"]):
            return "narrative_prompt_mapped_to_non_narrative_path"
    if ("date" in label or label in {"dos", "dob"}) and not is_date_like_path(path):
        return "date_label_mapped_to_non_date_path"
    return None


def build_mapping_from_llm(
    pdf_path: Path,
    schema_paths: list[str],
    candidates: list[RectCandidate],
    llm_mapping: dict[str, Any],
    template_id: str | None,
) -> dict[str, Any]:
    fingerprint = compute_fingerprint(pdf_path)
    by_id = {i: cand for i, cand in enumerate(candidates, 1)}
    seen: dict[str, int] = {}
    fields: list[dict[str, Any]] = []
    date_fields: list[dict[str, Any]] = []
    checkbox_groups: list[dict[str, Any]] = []
    allowed_paths = set(schema_paths)

    for item in llm_mapping.get("fields", []):
        cand = by_id.get(int(item.get("candidateId", 0)))
        if not cand or cand.kind != "text":
            continue
        json_path = item.get("jsonPath")
        if json_path not in allowed_paths:
            json_path = heuristic_json_path(cand.label, cand.sectionPath, schema_paths)
        field = {
            "fieldId": unique_id(str(item.get("fieldId") or cand.label), seen),
            "type": item.get("type") or "text",
            "jsonPath": json_path,
            "page": cand.page,
            "sectionPath": cand.sectionPath,
            "label": cand.label,
            "labelRect": cand.labelRect,
            "fillRect": cand.fillRect,
            "fontSize": 9,
            "overflow": "shrink_then_clip",
            "confidence": cand.confidence,
            "nearbyText": cand.nearbyText,
            "review": "ok" if json_path else "needs_json_path",
        }
        if field["type"] == "date":
            field.update({"format": item.get("format") or "MM/DD/YYYY", "mode": "single", "separators": "draw"})
            add_transform_metadata(field, schema_paths, "date")
            reason = production_skip_reason(field, "date")
            if reason:
                field["review"] = reason
            date_fields.append(field)
        else:
            add_transform_metadata(field, schema_paths, "text")
            reason = production_skip_reason(field, "field")
            if reason:
                field["review"] = reason
            fields.append(field)

    for group in llm_mapping.get("checkboxGroups", []):
        options = []
        page = None
        for option in group.get("options", []):
            cand = by_id.get(int(option.get("candidateId", 0)))
            if not cand or cand.kind != "checkbox":
                continue
            page = cand.page
            options.append({"value": option.get("value") or "true", "label": option.get("label") or cand.nearbyText, "boxRect": cand.fillRect})
        if not options:
            continue
        json_path = group.get("jsonPath")
        if json_path not in allowed_paths:
            json_path = None
        checkbox_groups.append(
            {
                "fieldId": unique_id(str(group.get("fieldId") or "checkbox_group"), seen),
                "type": "checkbox_group",
                "jsonPath": json_path,
                "page": page,
                "sectionPath": [],
                "label": " / ".join(o["label"] for o in options)[:240],
                "options": options,
                "confidence": 0.8,
                "multiSelect": False,
                "matchMode": "contains_normalized",
                "review": "ok" if json_path else "needs_json_path",
            }
        )

    return normalize_mapping_payload({
        "templateId": template_id or slug(pdf_path.stem, "template"),
        "templateVersion": "v1",
        "templateFingerprint": fingerprint,
        "pageCount": fingerprint["pageCount"],
        "pageSizes": fingerprint["pageSizes"],
        "fields": fields,
        "checkboxGroups": checkbox_groups,
        "dateFields": date_fields,
        "repeaters": llm_mapping.get("repeaters", []),
        "createdBy": "pdf_fill_engine.create_mapping",
        "reviewStatus": "draft_needs_review",
        "schemaPaths": schema_paths,
        "candidateCount": len(candidates),
        "mappingMode": "llm",
    })


def build_mapping(
    pdf_path: Path,
    schema_path: Path,
    out_path: Path,
    template_id: str | None = None,
    use_llm: bool = False,
    model: str = "gpt-4.1-mini",
    env_path: Path | None = None,
) -> dict[str, Any]:
    schema = load_json(schema_path)
    schema_paths = extract_json_paths(schema)
    fingerprint = compute_fingerprint(pdf_path)
    candidates = build_candidates(pdf_path)

    if use_llm:
        llm_mapping = llm_select_mapping(pdf_path, candidates, schema_paths, model, env_path)
        mapping = build_mapping_from_llm(pdf_path, schema_paths, candidates, llm_mapping, template_id)
        write_json(out_path, mapping)
        return mapping

    fields = []
    checkbox_groups = []
    date_fields = []
    for idx, cand in enumerate(candidates, 1):
        field_id = slug("_".join(cand.sectionPath + [cand.label]), f"field_{idx}")
        if cand.kind == "checkbox":
            checkbox_groups.append(
                {
                    "fieldId": f"{field_id}_{idx}",
                    "type": "checkbox_group",
                    "jsonPath": None,
                    "page": cand.page,
                    "sectionPath": cand.sectionPath,
                    "label": cand.nearbyText,
                    "options": [
                        {
                            "value": "true",
                            "label": cand.nearbyText[:160],
                            "boxRect": cand.fillRect,
                        }
                    ],
                    "confidence": cand.confidence,
                    "multiSelect": False,
                    "matchMode": "contains_normalized",
                    "review": "needs_json_path_and_option_value",
                }
            )
            continue
        json_path = heuristic_json_path(cand.label, cand.sectionPath, schema_paths)
        is_date = any(term in cand.label.lower() for term in ["date", "dob", "dos"])
        item = {
            "fieldId": field_id,
            "type": "date" if is_date else "text",
            "jsonPath": json_path,
            "page": cand.page,
            "sectionPath": cand.sectionPath,
            "label": cand.label,
            "labelRect": cand.labelRect,
            "fillRect": cand.fillRect,
            "fontSize": 9,
            "overflow": "shrink_then_clip",
            "confidence": cand.confidence,
            "nearbyText": cand.nearbyText,
            "review": "ok" if json_path else "needs_json_path",
        }
        if is_date:
            item.update({"format": "MM/DD/YYYY", "mode": "single", "separators": "draw"})
            add_transform_metadata(item, schema_paths, "date")
            reason = production_skip_reason(item, "date")
            if reason:
                item["review"] = reason
            date_fields.append(item)
        else:
            add_transform_metadata(item, schema_paths, "text")
            reason = production_skip_reason(item, "field")
            if reason:
                item["review"] = reason
            fields.append(item)

    mapping = normalize_mapping_payload({
        "templateId": template_id or slug(pdf_path.stem, "template"),
        "templateVersion": "v1",
        "templateFingerprint": fingerprint,
        "pageCount": fingerprint["pageCount"],
        "pageSizes": fingerprint["pageSizes"],
        "fields": fields,
        "checkboxGroups": checkbox_groups,
        "dateFields": date_fields,
        "repeaters": [],
        "createdBy": "pdf_fill_engine.create_mapping",
        "reviewStatus": "draft_needs_review",
        "schemaPaths": schema_paths,
        "candidateCount": len(candidates),
        "mappingMode": "heuristic",
    })
    write_json(out_path, mapping)
    return mapping


In [ ]:
mapping = build_mapping(
    pdf_path=PDF_PATH,
    schema_path=DATA_JSON_PATH,
    out_path=MAPPING_PATH,
    template_id=TEMPLATE_ID,
    use_llm=USE_LLM_MAPPING,
    model=OPENAI_MODEL,
    env_path=ENV_FILE,
)

print(json.dumps({
    "templateId": mapping["templateId"],
    "reviewStatus": mapping["reviewStatus"],
    "mappingMode": mapping.get("mappingMode"),
    "candidateCount": mapping["candidateCount"],
    "fields": len(mapping.get("fields", [])),
    "dateFields": len(mapping.get("dateFields", [])),
    "checkboxGroups": len(mapping.get("checkboxGroups", [])),
}, indent=2))

## 6. Mapping Review Table

This is the human-review checkpoint for a new or changed template.

### What this table shows

For each mapped item, the notebook prints:

- Collection: `fields`, `dateFields`, or `checkboxGroups`.
- `fieldId`: stable identifier used by diagnostics.
- `label`: source label from the PDF.
- `jsonPath`: source value path from the DB JSON.
- `valueTransform`: deterministic renderer such as `name_full`, `date`, `date_range`, `currency`, or `text`.
- `review`: production trust status.
- Page number.

### Review status interpretation

- `ok`: eligible for production fill.
- `needs_json_path`: field exists but no safe source path is known.
- `needs_json_path_and_option_value`: checkbox needs a source path and option definitions.
- `non_fillable_static_text`: label is static destination/instruction text, not a fill target.
- `option_list_not_text_field`: checkbox/option label should not receive narrative text.
- `unsafe_signature_mapping`: signature was not mapped to a real signature source.
- `date_label_mapped_to_non_date_path`: date label pointed to a non-date value.

### What business users should review

Focus on high-risk values: names, member IDs, claim numbers, service dates, amounts, checkboxes, and long narrative fields.

### Expected output

A review row count and a row-by-row JSON summary.

In [ ]:
review_rows = []
for collection in ["fields", "dateFields", "checkboxGroups"]:
    for item in mapping.get(collection, []):
        review_rows.append({
            "collection": collection,
            "fieldId": item.get("fieldId"),
            "label": item.get("label"),
            "jsonPath": item.get("jsonPath"),
            "valueTransform": item.get("valueTransform"),
            "review": item.get("review"),
            "page": item.get("page"),
        })

ok_count = sum(1 for row in review_rows if row["review"] == "ok")
print(f"Review rows: {len(review_rows)} | production-ok: {ok_count} | skipped/review-needed: {len(review_rows) - ok_count}")

for row in review_rows[:60]:
    print(json.dumps(row, ensure_ascii=False))

## 7. Runtime Formatters, Validators, And Drawing Engine

This cell contains the deterministic production runtime. Mapping artifacts and fill diagnostics are validated with Pydantic models before they are written, loaded, or returned.

### Value safety

The runtime refuses to draw values when:

- `jsonPath` is missing or invalid.
- `review != "ok"`.
- The resolved value is empty, a dict, a list, or the whole input object.
- A date cannot be parsed.
- The field is semantically unsafe for production fill.

### Supported value transforms

- `text`: normalize whitespace and draw a scalar value.
- `name_full`: compose first/middle/last name cleanly.
- `date`: normalize `YYYYMMDD`, `YYYY-MM-DD`, or `MM/DD/YYYY` to `MM/DD/YYYY`.
- `date_range`: render `MM/DD/YYYY - MM/DD/YYYY` from start/end dates.
- `currency`: render numeric strings as `1234.56` unless a mapping requests a dollar sign.

### Layout safety

Before drawing, the runtime sanitizes every fill rectangle:

- Shift value start after the label.
- Treat existing PDF text as protected.
- Trim before helper text or neighboring labels.
- Clip drawn text to the rectangle.
- Skip the field if no safe writable area remains.

### Diagnostics

Every fill produces diagnostics showing filled fields, skipped fields, skipped reasons, warnings, and rendered rectangle previews.

In [ ]:
def parse_date(value: Any) -> str | None:
    if value is None or value == "":
        return None
    text = str(value).strip()
    for fmt in DATE_INPUT_FORMATS:
        try:
            return datetime.strptime(text, fmt).strftime("%m/%d/%Y")
        except ValueError:
            pass
    return None


def format_currency(value: Any, currency_symbol: bool = False) -> str | None:
    if value is None or isinstance(value, (dict, list)):
        return None
    text = normalize_space(value).replace(",", "")
    if not text:
        return None
    try:
        rendered = f"{float(text):.2f}"
    except ValueError:
        return text
    return f"${rendered}" if currency_symbol else rendered


def infer_name_base_path(path: str | None) -> str | None:
    if not path:
        return None
    for base in ["result.patient", "result.subscriber", "result.dependent"]:
        if path == base or path.startswith(base + "."):
            return base
    return None


def format_full_name(data: Any, path: str | None) -> str | None:
    base = infer_name_base_path(path)
    if not base:
        value = resolve_json_path(data, path)
        if isinstance(value, str):
            return normalize_space(value) or None
        return None
    parts = [
        resolve_json_path(data, f"{base}.firstName"),
        resolve_json_path(data, f"{base}.middleName"),
        resolve_json_path(data, f"{base}.lastName"),
    ]
    rendered = " ".join(normalize_space(part) for part in parts if normalize_space(part))
    return rendered or None


def format_date_range(start: Any, end: Any) -> str | None:
    first = parse_date(start)
    second = parse_date(end)
    if first and second and first != second:
        return f"{first} - {second}"
    return first or second


def scalar_text(value: Any) -> str | None:
    if value is None or isinstance(value, (dict, list)):
        return None
    text = normalize_space(value)
    return text or None


def preview_value(value: Any, limit: int = 120) -> str:
    text = normalize_space(value)
    return text if len(text) <= limit else text[: limit - 3] + "..."


def format_mapped_value(data: Any, field: dict[str, Any], default_transform: str = "text") -> tuple[str | None, str | None]:
    path = field.get("jsonPath")
    if not is_nonempty_json_path(path):
        return None, "missing_json_path"
    transform = field.get("valueTransform") or infer_value_transform(
        str(field.get("label") or ""),
        str(field.get("fieldId") or ""),
        path,
        default_transform,
    )
    value = resolve_json_path(data, path)
    if value is data or isinstance(value, (dict, list)):
        return None, "non_scalar_value"
    if value is None or value == "":
        return None, "empty_value"

    if transform == "name_full":
        rendered = format_full_name(data, path)
    elif transform == "date":
        rendered = parse_date(value)
        if not rendered:
            return None, "invalid_date"
    elif transform == "date_range":
        secondary = field.get("jsonPathSecondary")
        end_value = resolve_json_path(data, secondary) if is_nonempty_json_path(secondary) else None
        rendered = format_date_range(value, end_value)
        if not rendered:
            return None, "invalid_date_range"
    elif transform == "currency":
        rendered = format_currency(value, bool(field.get("currencySymbol")))
    else:
        rendered = scalar_text(value)

    if not rendered:
        return None, "empty_rendered_value"
    return rendered, None


def valid_rect(rect: Any) -> bool:
    if not isinstance(rect, list) or len(rect) != 4:
        return False
    try:
        x1, y1, x2, y2 = [float(v) for v in rect]
    except (TypeError, ValueError):
        return False
    return x2 > x1 and y2 > y1


def validate_mapping(mapping: dict[str, Any], schema_paths: list[str]) -> dict[str, list[str]]:
    errors: list[str] = []
    warnings: list[str] = []
    try:
        mapping = normalize_mapping_payload(mapping)
    except ValidationError as exc:
        return MappingValidationResult(errors=pydantic_error_messages(exc), warnings=[]).model_dump(mode="json")

    allowed_paths = set(schema_paths)
    seen_ids: set[str] = set()
    rects_by_page: dict[int, list[tuple[str, list[float]]]] = {}

    def check_id(item: dict[str, Any], collection: str) -> None:
        field_id = str(item.get("fieldId") or "")
        if not field_id:
            errors.append(f"{collection}: missing fieldId")
        elif field_id in seen_ids:
            errors.append(f"duplicate fieldId: {field_id}")
        else:
            seen_ids.add(field_id)

    def check_path(item: dict[str, Any], collection: str) -> None:
        field_id = item.get("fieldId", "<unknown>")
        path = item.get("jsonPath")
        if item.get("review") == "ok" and not is_nonempty_json_path(path):
            warnings.append(f"{collection}.{field_id}: review ok but jsonPath is missing")
        if is_nonempty_json_path(path) and path not in allowed_paths:
            warnings.append(f"{collection}.{field_id}: jsonPath not found in schemaPaths: {path}")
        secondary = item.get("jsonPathSecondary")
        if is_nonempty_json_path(secondary) and secondary not in allowed_paths:
            warnings.append(f"{collection}.{field_id}: jsonPathSecondary not found in schemaPaths: {secondary}")

    def add_rect(item: dict[str, Any], rect_key: str, collection: str) -> None:
        field_id = str(item.get("fieldId") or collection)
        rect = item.get(rect_key)
        if not valid_rect(rect):
            errors.append(f"{collection}.{field_id}: invalid {rect_key}")
            return
        page = int(item.get("page") or 0)
        if page <= 0:
            errors.append(f"{collection}.{field_id}: invalid page")
            return
        rects_by_page.setdefault(page, []).append((field_id, [float(v) for v in rect]))

    for collection in ["fields", "dateFields"]:
        for item in mapping.get(collection, []):
            check_id(item, collection)
            check_path(item, collection)
            add_rect(item, "fillRect", collection)
            reason = production_skip_reason(item, "date" if collection == "dateFields" else "field")
            if reason and item.get("review") == "ok":
                warnings.append(f"{collection}.{item.get('fieldId')}: {reason}")
            if collection == "dateFields" and not item.get("format"):
                warnings.append(f"{collection}.{item.get('fieldId')}: date field is missing format")
            if collection == "dateFields" and item.get("valueTransform") not in {None, "date", "date_range"}:
                warnings.append(f"{collection}.{item.get('fieldId')}: date field has non-date transform")

    for group in mapping.get("checkboxGroups", []):
        check_id(group, "checkboxGroups")
        check_path(group, "checkboxGroups")
        if group.get("review") == "ok" and not is_nonempty_json_path(group.get("jsonPath")):
            warnings.append(f"checkboxGroups.{group.get('fieldId')}: review ok but jsonPath is missing")
        if not group.get("options"):
            warnings.append(f"checkboxGroups.{group.get('fieldId')}: no options")
        for index, option in enumerate(group.get("options", []), 1):
            rect = option.get("boxRect")
            if not valid_rect(rect):
                errors.append(f"checkboxGroups.{group.get('fieldId')}.option{index}: invalid boxRect")

    for page, items in rects_by_page.items():
        for i, (left_id, left_rect) in enumerate(items):
            for right_id, right_rect in items[i + 1:]:
                if rect_overlap(left_rect, right_rect) > 0.65:
                    warnings.append(f"overlapping fill rects on page {page}: {left_id} and {right_id}")

    return MappingValidationResult(errors=errors, warnings=warnings).model_dump(mode="json")


def fit_text(c: canvas.Canvas, text: str, rect: list[float], font_size: float) -> float:
    x1, _y1, x2, _y2 = rect
    size = font_size
    while size > 5 and stringWidth(text, "Helvetica", size) > (x2 - x1):
        size -= 0.5
    c.setFont("Helvetica", size)
    return size


def draw_text_field(c: canvas.Canvas, value: Any, rect: list[float], font_size: float = 9, overflow: str = "shrink_then_clip") -> bool:
    if value is None or value == "":
        return False
    text = str(value)
    x1, y1, x2, y2 = rect
    if "\n" in text or len(text) > 80 or (y2 - y1) > 24:
        c.saveState()
        path = c.beginPath()
        path.rect(x1, y1, max(1, x2 - x1), max(1, y2 - y1))
        c.clipPath(path, stroke=0, fill=0)
        c.setFont("Helvetica", font_size)
        max_chars = max(10, int((x2 - x1) / (font_size * 0.48)))
        lines: list[str] = []
        for para in text.splitlines() or [text]:
            lines.extend(wrap(para, max_chars) or [""])
        line_height = font_size + 2
        y = y2 - font_size
        drawn = 0
        for line in lines:
            if y < y1 + 2:
                break
            c.drawString(x1 + 1, y, line)
            y -= line_height
            drawn += 1
        c.restoreState()
        return drawn < len(lines)
    c.saveState()
    path = c.beginPath()
    path.rect(x1, y1, max(1, x2 - x1), max(1, y2 - y1))
    c.clipPath(path, stroke=0, fill=0)
    size = fit_text(c, text, rect, font_size)
    c.drawString(x1 + 1, y1 + max(2, (y2 - y1 - font_size) / 2), text)
    c.restoreState()
    return stringWidth(text, "Helvetica", size) > (x2 - x1)


def draw_check(c: canvas.Canvas, rect: list[float]) -> None:
    x1, y1, x2, y2 = rect
    c.setStrokeColor(black)
    c.setLineWidth(1.2)
    c.line(x1 + 1, y1 + 1, x2 - 1, y2 - 1)
    c.line(x1 + 1, y2 - 1, x2 - 1, y1 + 1)


def should_check(value: Any, option: dict[str, Any]) -> bool:
    if value is None:
        return False
    if isinstance(value, bool):
        return value
    text = normalize_space(value).lower()
    targets = [normalize_space(option.get("value", "")).lower(), normalize_space(option.get("label", "")).lower()]
    return any(target and target in text for target in targets)


def fill_pdf(pdf_path: Path, mapping_path: Path, data_path: Path, output_path: Path, allow_drift: bool = False) -> dict[str, Any]:
    try:
        mapping = normalize_mapping_payload(load_json(mapping_path))
    except ValidationError as exc:
        raise RuntimeError("mapping_invalid: " + "; ".join(pydantic_error_messages(exc))) from exc
    data = load_json(data_path)
    current_fp = compute_fingerprint(pdf_path)
    expected_fp = mapping.get("templateFingerprint", {})
    drift = expected_fp.get("value") != current_fp.get("value")
    if drift and not allow_drift:
        raise RuntimeError("mapping_stale: PDF fingerprint does not match mapping. Use --allow-drift to override.")

    schema_paths = mapping.get("schemaPaths") or extract_json_paths(data)
    apply_runtime_backfills(mapping, schema_paths)
    try:
        mapping = normalize_mapping_payload(mapping)
    except ValidationError as exc:
        raise RuntimeError("mapping_invalid_after_backfill: " + "; ".join(pydantic_error_messages(exc))) from exc
    validation = validate_mapping(mapping, schema_paths)
    if validation["errors"]:
        raise RuntimeError("mapping_invalid: " + "; ".join(validation["errors"]))

    diagnostics: dict[str, Any] = {
        "filledFields": [],
        "skippedFields": [],
        "skippedReasons": {},
        "warnings": list(validation["warnings"]),
    }

    def skip(field: dict[str, Any], reason: str) -> None:
        field_id = str(field.get("fieldId") or field.get("label") or "<unknown>")
        diagnostics["skippedFields"].append(
            {
                "fieldId": field_id,
                "reason": reason,
                "jsonPath": field.get("jsonPath"),
            }
        )
        diagnostics["skippedReasons"][reason] = diagnostics["skippedReasons"].get(reason, 0) + 1

    def filled(field: dict[str, Any], value: Any, kind: str, truncated: bool = False, rect: list[float] | None = None) -> None:
        item = {
            "fieldId": field.get("fieldId"),
            "type": kind,
            "jsonPath": field.get("jsonPath"),
            "renderedValuePreview": preview_value(value),
        }
        if rect:
            item["renderedRect"] = rect
        if truncated:
            item["truncated"] = True
            diagnostics["warnings"].append(f"{field.get('fieldId')}: rendered value was truncated to fit fillRect")
        diagnostics["filledFields"].append(item)

    reader = PdfReader(str(pdf_path))
    text_runs = extract_text_runs_pymupdf(pdf_path) or extract_text_runs_pypdf(reader)
    with tempfile.TemporaryDirectory(prefix="pdf_fill_") as tmp:
        overlay = Path(tmp) / "overlay.pdf"
        first = reader.pages[0].mediabox
        c = canvas.Canvas(str(overlay), pagesize=(float(first.width), float(first.height)))
        by_page: dict[int, list[tuple[str, dict[str, Any]]]] = {}
        for field in mapping.get("fields", []):
            if field.get("page"):
                by_page.setdefault(int(field["page"]), []).append(("field", field))
        for field in mapping.get("dateFields", []):
            if field.get("page"):
                by_page.setdefault(int(field["page"]), []).append(("date", field))
        for group in mapping.get("checkboxGroups", []):
            page = None
            for option in group.get("options", []):
                rect = option.get("boxRect")
                if rect:
                    page = group.get("page") or option.get("page")
                    break
            if page:
                by_page.setdefault(int(page), []).append(("checkbox_group", group))

        for page_index, page in enumerate(reader.pages):
            c.setPageSize((float(page.mediabox.width), float(page.mediabox.height)))
            for kind, field in by_page.get(page_index + 1, []):
                if kind in {"field", "date"}:
                    if field.get("review") != "ok":
                        skip(field, "review_not_ok")
                        continue
                    semantic_reason = production_skip_reason(field, kind)
                    if semantic_reason:
                        skip(field, semantic_reason)
                        continue
                    value, reason = format_mapped_value(data, field, "date" if kind == "date" else "text")
                    if reason:
                        skip(field, reason)
                        if reason in {"invalid_date", "invalid_date_range"}:
                            diagnostics["warnings"].append(f"{field.get('fieldId')}: {reason} for {field.get('jsonPath')}")
                        continue
                    safe_rect, layout_warnings = sanitize_fill_rect(field, float(page.mediabox.width), text_runs)
                    for warning in layout_warnings:
                        diagnostics["warnings"].append(f"{field.get('fieldId')}: {warning}")
                    if not safe_rect:
                        skip(field, "layout_no_safe_fill_rect")
                        continue
                    truncated = draw_text_field(c, value, safe_rect, field.get("fontSize", 9), field.get("overflow", "shrink_then_clip"))
                    filled(field, value, kind, truncated, safe_rect)
                elif kind == "checkbox_group":
                    if field.get("review") != "ok":
                        skip(field, "review_not_ok")
                        continue
                    if not is_nonempty_json_path(field.get("jsonPath")):
                        skip(field, "missing_json_path")
                        continue
                    value = resolve_json_path(data, field.get("jsonPath"))
                    if value is None or value == "" or isinstance(value, (dict, list)):
                        skip(field, "empty_or_non_scalar_value")
                        continue
                    matched = False
                    for option in field.get("options", []):
                        if should_check(value, option):
                            draw_check(c, option["boxRect"])
                            matched = True
                            if not field.get("multiSelect", False):
                                break
                    if matched:
                        filled(field, value, kind)
                    else:
                        diagnostics["warnings"].append(f"{field.get('fieldId')}: no checkbox option matched value '{preview_value(value)}'")
                        skip(field, "checkbox_no_match")
            c.showPage()
        c.save()

        writer = PdfWriter()
        overlay_reader = PdfReader(str(overlay))
        for i, page in enumerate(reader.pages):
            page.merge_page(overlay_reader.pages[i])
            writer.add_page(page)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("wb") as fh:
            writer.write(fh)

    return FillResult(
        output=str(output_path),
        mappingDrift=drift,
        fields=len(mapping.get("fields", [])),
        dateFields=len(mapping.get("dateFields", [])),
        checkboxGroups=len(mapping.get("checkboxGroups", [])),
        diagnostics=diagnostics,
    ).model_dump(mode="json", exclude_none=False)


## 8. Validate Mapping Before Fill

This cell validates the mapping artifact before any ink is drawn onto the PDF.

### Structural errors

These stop the fill:

- Duplicate `fieldId`.
- Invalid or missing `fillRect`.
- Invalid page numbers.
- Invalid checkbox `boxRect`.

### Warnings

These are reported but do not necessarily stop safe fields from filling:

- `review == "ok"` with a missing path.
- Paths not found in schema paths.
- Overlapping fill rectangles.
- Date fields missing expected date metadata.
- Semantic safety concerns.

### What to do if validation fails

Fix the mapping artifact or regenerate it from the updated PDF template. Do not bypass validation for production.

### Expected output

A JSON object with `errors` and `warnings`.

In [ ]:
mapping = load_json(MAPPING_PATH)
apply_runtime_backfills(mapping, mapping.get("schemaPaths") or schema_paths)
validation = validate_mapping(mapping, mapping.get("schemaPaths") or schema_paths)

print(json.dumps(validation, indent=2))
if validation["errors"]:
    raise RuntimeError("Mapping has structural errors. Fix the mapping before filling.")

## 9. Fill The PDF Deterministically

This is the production fill step.

### Inputs

- Original PDF template.
- Mapping artifact.
- DB JSON payload.

### Runtime behavior

- Verify PDF fingerprint unless `allow_drift=True` is explicitly set.
- Apply safe legacy backfills for known semantic cases.
- Validate mapping structure.
- Format each value deterministically.
- Sanitize the final rectangle.
- Draw text or checkbox marks directly onto an overlay.
- Merge the overlay with the original PDF.
- Write a flattened PDF.

### Important rule

No GPT call is made here. Production filling is deterministic and auditable.

### Expected output

A result object containing the output path, mapping drift status, field counts, and diagnostics.

In [ ]:
fill_result = fill_pdf(
    pdf_path=PDF_PATH,
    mapping_path=MAPPING_PATH,
    data_path=DATA_JSON_PATH,
    output_path=FILLED_PDF_PATH,
    allow_drift=False,
)

print(json.dumps(fill_result, indent=2))

## 10. Diagnostics Review

Diagnostics explain exactly what happened during fill.

### Filled fields

Each filled field includes:

- `fieldId`
- field type
- `jsonPath`
- `renderedValuePreview`
- `renderedRect`
- optional `truncated` flag

### Skipped fields

Skipped fields include the field ID, JSON path, and skip reason.

Common skip reasons:

- `review_not_ok`
- `missing_json_path`
- `empty_value`
- `invalid_date`
- `non_scalar_value`
- `layout_no_safe_fill_rect`
- `checkbox_no_match`

### How to use diagnostics operationally

Diagnostics should feed the template review workflow. A skipped field is not necessarily a bug; often it is the correct fail-safe behavior until the mapping is reviewed.

### Expected output

A readable audit trail of filled fields, skipped reasons, and warnings.

In [ ]:
diagnostics = fill_result["diagnostics"]

print("Filled fields:", len(diagnostics["filledFields"]))
for item in diagnostics["filledFields"]:
    print(json.dumps(item, ensure_ascii=False))

print("\nSkipped reasons:")
print(json.dumps(diagnostics["skippedReasons"], indent=2))

print("\nWarnings:")
for warning in diagnostics["warnings"]:
    print("-", warning)

## 11. PDF Preservation And Smoke Checks

This stage performs basic machine checks on the generated PDF.

### What is checked

- Output file exists.
- Output PDF is readable.
- Page count is unchanged.
- Page boxes are unchanged.
- The filled PDF does not contain a dumped JSON payload.
- Extracted text length is nonzero.

### What this does not check

These checks do not replace visual QA. They catch severe runtime failures but cannot guarantee that every value is visually perfect.

### Recommended QA practice

For a new template, review the filled PDF visually and compare diagnostics with the mapping review table before promoting the mapping to production.

In [ ]:
source_reader = PdfReader(str(PDF_PATH))
filled_reader = PdfReader(str(FILLED_PDF_PATH))
filled_text = "\n".join(page.extract_text() or "" for page in filled_reader.pages)

checks = {
    "outputExists": FILLED_PDF_PATH.exists(),
    "readable": len(filled_reader.pages) > 0,
    "pageCountSame": len(source_reader.pages) == len(filled_reader.pages),
    "pageBoxesSame": [[float(p.mediabox.width), float(p.mediabox.height)] for p in source_reader.pages]
        == [[float(p.mediabox.width), float(p.mediabox.height)] for p in filled_reader.pages],
    "noJsonPayloadDump": "appealInventoryId" not in filled_text and "claimStatusToken" not in filled_text,
    "filledTextLength": len(filled_text),
}
print(json.dumps(checks, indent=2))

## 12. Optional: Reuse An Existing Mapping

Use this section after a mapping has already been reviewed and accepted for a payer template/version.

### Production pattern

Create mapping once:

```text
PDF template + schema JSON -> reviewed mapping artifact
```

Reuse mapping many times:

```text
PDF template + reviewed mapping artifact + claim JSON -> filled PDF
```

### When to create a new mapping version

Create a new mapping if:

- The PDF layout changes.
- The PDF text changes materially.
- The payer releases a new form version.
- A QA review finds a field that needs correction.

### Why fingerprinting matters

The fill step compares the current PDF fingerprint against the mapping fingerprint. If they differ, the mapping may be stale and production fill should stop unless explicitly overridden.

In [ ]:
# Example:
# REVIEWED_MAPPING_PATH = Path("outputs/AETNA_Form1_template_mapping_layout_safe.json")
# NEW_DATA_JSON_PATH = Path("database_schema.json")
# NEW_OUTPUT_PATH = Path("outputs/AETNA_Form1_production_fill.pdf")
#
# result = fill_pdf(PDF_PATH, REVIEWED_MAPPING_PATH, NEW_DATA_JSON_PATH, NEW_OUTPUT_PATH)
# print(json.dumps(result, indent=2))

## 13. Handoff Notes

This notebook is both a technical demo and an onboarding artifact.

### Recommended production workflow

1. Create a draft mapping for each payer PDF template/version.
2. Review mapping rows, especially fields not marked `ok`.
3. Correct or approve mappings as needed.
4. Store the reviewed mapping artifact with the template version and fingerprint.
5. Run deterministic fill for each DB JSON payload.
6. Monitor diagnostics and warnings as an operations review queue.

### What the current implementation handles well

- Duplicate labels by using section context and geometry.
- Full-name composition.
- Date normalization and date ranges.
- Currency formatting.
- Safe handling of missing/invalid JSON paths.
- Avoiding label/helper-text overlap.
- Skipping uncertain fields instead of drawing wrong values.

### Current limitations

- Checkbox groups remain conservative unless a clear JSON source and option mapping exists.
- Tables/repeating service lines need template-specific repeater metadata before production use.
- If a form leaves no safe writable area for a value, the field is skipped.
- Visual QA is still required before a new template mapping is promoted.

### Business takeaway

The product is not trying to guess at runtime. It creates a reusable, reviewable template mapping and then fills forms deterministically with traceable diagnostics.